<a href="https://colab.research.google.com/github/Subuktageen-Farooqi/ms_course_deeplearning/blob/main/ms_deeplearning_tutorial_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 04: Image Augmentation

Step 1: Importing Necessary Libraries

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array, load_img
import numpy as np
import os
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

Step 2: Define the Folder Where Augmented Images Will Be Saved

In [ ]:
output_dir = input("Please enter the path to the folder to save augmented images: ")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

Step 3: Initialize the ImageDataGenerator with Augmentation Parameters

In [ ]:
datagen = ImageDataGenerator(
    rotation_range=40,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.5, 1.5]
)

Step 4: Load the Image and Convert It to a NumPy Array

In [ ]:
img = input("Please enter the path to the source image: ")
x = img_to_array(img)
cv2_imshow(x)

Step 5: Reshape the Image for Batch Processing

In [ ]:
x = np.expand_dims(x, axis=0)

Step 6: Generate and Save Augmented Images

In [ ]:
i = 0
for batch in datagen.flow(x,
                          batch_size=1,
                          save_to_dir=output_dir,
                          save_prefix='aug',
                          save_format='jpeg'):
    i += 1
    if i >= 40:
        break

Step 7: Confirm the Saved Location of Augmented Images

In [ ]:
print(f"Augmented images saved in: {os.path.abspath(output_dir)}")

Augmented images saved in: /content/augmented_images


Task: Bulk Image Augmentation

In [ ]:
# Refactored Pipeline for Image Augmentation

# Ensure PIL.Image is imported for saving augmented images
from PIL import Image

# Step 1: Prompt user for input folder path
input_dir = input("Please enter the path to the folder containing your source images: ")

# Validate the input directory
if not os.path.exists(input_dir):
    print(f"Error: Input directory '{input_dir}' does not exist.")
else:
    print(f"Input directory set to: {input_dir}")

    # Ensure output_dir is defined and exists (from previous cells)
    # output_dir is already defined in cell 68fc72eJuAAW
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # datagen is already initialized in cell iJ02kQe-uSxC

    image_files = []
    for f in os.listdir(input_dir):
        # Basic check for common image extensions.
        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff')):
            image_files.append(os.path.join(input_dir, f))

    if not image_files:
        print(f"No image files found in '{input_dir}'. Please check the path and file types.")
    else:
        print(f"Found {len(image_files)} image(s) in the input directory.")

        total_augmented_images_generated = 0
        for source_img_idx, img_path in enumerate(image_files):
            print(f"\nProcessing source image {source_img_idx + 1}/{len(image_files)}: {os.path.basename(img_path)}")

            # Load the image and convert it to a NumPy array
            img = load_img(img_path)
            x = img_to_array(img)
            # Reshape the image for batch processing (add a dimension for batch size)
            x = np.expand_dims(x, axis=0)

            # Generate and save 40 augmented images for the current source image
            i = 0 # Counter for augmented images per source image
            for batch in datagen.flow(x, batch_size=1): # Generate one augmented image at a time
                augmented_image_array = batch[0] # Get the first (and only) image from the batch

                # Convert numpy array back to PIL Image to save
                # Ensure the array is uint8 type before converting to PIL Image
                augmented_pil_img = Image.fromarray(augmented_image_array.astype('uint8'))

                # Naming convention: aug_img_<source_image_number>_<augmented_image_number_out_of_40>.jpg
                filename = f"aug_img_{source_img_idx}_{i}.jpg"
                save_path = os.path.join(output_dir, filename)

                augmented_pil_img.save(save_path)

                i += 1
                total_augmented_images_generated += 1
                if i >= 40: # Generate 40 augmented images per source image
                    break

        print(f"\nSuccessfully generated {total_augmented_images_generated} augmented images in '{output_dir}'.")


Please enter the path to the folder containing your source images: /content/drive/MyDrive/MS_DeepLearning_Course/source_images
Input directory set to: /content/drive/MyDrive/MS_DeepLearning_Course/source_images
Found 10 image(s) in the input directory.

Processing source image 1/10: pexels-richaa-benny-2148463131-30110819.jpg

Processing source image 2/10: pexels-anjali-satasiya-1338760943-25857898.jpg

Processing source image 3/10: pexels-crgrandbois-9068593.jpg

Processing source image 4/10: pexels-magda-ehlers-pexels-34844375.jpg

Processing source image 5/10: pexels-vikas-bhriguvanshi-520057-4518109.jpg

Processing source image 6/10: pexels-szafran-34985885.jpg


Task: PyTorch Implementation

In [ ]:
from pathlib import Path

from PIL import Image, UnidentifiedImageError
from torchvision import transforms

# Configuration
OUTPUT_DIR = Path("augmented_images")
NUM_AUGMENTATIONS = 40
OUTPUT_PREFIX = "single_aug"
JPEG_QUALITY = 95

# Augmentation settings (tutorial-style)
ROTATION_DEGREES = 35
AFFINE_DEGREES = 12
AFFINE_TRANSLATE = (0.08, 0.08)
AFFINE_SCALE = (0.85, 1.15)
AFFINE_SHEAR = 15
RESIZED_CROP_SCALE = (0.8, 1.0)
RESIZED_CROP_RATIO = (0.9, 1.1)
H_FLIP_PROBABILITY = 0.5
BRIGHTNESS = 0.3
CONTRAST = 0.2
SATURATION = 0.2


def build_augmentation_pipeline(target_size: tuple[int, int]) -> transforms.Compose:
    """Create a stochastic augmentation pipeline for one image."""
    return transforms.Compose(
        [
            transforms.RandomRotation(degrees=ROTATION_DEGREES),
            transforms.RandomAffine(
                degrees=AFFINE_DEGREES,
                translate=AFFINE_TRANSLATE,
                scale=AFFINE_SCALE,
                shear=AFFINE_SHEAR,
            ),
            transforms.RandomResizedCrop(
                size=target_size,
                scale=RESIZED_CROP_SCALE,
                ratio=RESIZED_CROP_RATIO,
            ),
            transforms.RandomHorizontalFlip(p=H_FLIP_PROBABILITY),
            transforms.ColorJitter(
                brightness=BRIGHTNESS,
                contrast=CONTRAST,
                saturation=SATURATION,
            ),
        ]
    )


def augment_single_image(image_path: Path) -> int:
    """Generate and save augmented versions of one input image."""
    if not image_path.exists() or not image_path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")

    try:
        with Image.open(image_path) as source:
            original_image = source.convert("RGB")
    except (UnidentifiedImageError, OSError) as exc:
        raise ValueError(f"Unable to open image '{image_path}': {exc}") from exc

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    pipeline = build_augmentation_pipeline(original_image.size[::-1])

    for index in range(1, NUM_AUGMENTATIONS + 1):
        augmented = pipeline(original_image)
        output_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_{index}.jpg"
        augmented.save(output_path, format="JPEG", quality=JPEG_QUALITY)

    print(f"Source image path: {image_path.resolve()}")
    print(f"Output folder path: {OUTPUT_DIR.resolve()}")
    print(f"Total saved count: {NUM_AUGMENTATIONS}")

    return NUM_AUGMENTATIONS


def main() -> None:
    user_input = input("Enter the path to one image: ").strip()
    if not user_input:
        raise ValueError("No image path provided. Please enter a valid image file path.")

    augment_single_image(Path(user_input).expanduser())


if __name__ == "__main__":
    main()

Task: Bulk Image Augmentation with PyTorch

In [ ]:
"""
Bulk image data augmentation using PyTorch/torchvision.
Required packages: torch, torchvision, pillow
Run example: python bulk_image_augmentation_pytorch.py
"""

from pathlib import Path

from PIL import Image, UnidentifiedImageError
from torchvision import transforms

# Configuration
OUTPUT_DIR = Path("augmented_images")
NUM_AUGMENTATIONS_PER_IMAGE = 40
JPEG_QUALITY = 95
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Augmentation settings (tutorial-style)
ROTATION_DEGREES = 35
AFFINE_DEGREES = 12
AFFINE_TRANSLATE = (0.08, 0.08)
AFFINE_SCALE = (0.85, 1.15)
AFFINE_SHEAR = 15
RESIZED_CROP_SCALE = (0.8, 1.0)
RESIZED_CROP_RATIO = (0.9, 1.1)
H_FLIP_PROBABILITY = 0.5
BRIGHTNESS = 0.3
CONTRAST = 0.2
SATURATION = 0.2


def is_valid_image_file(path: Path) -> bool:
    """Check extension first to filter likely image files."""
    return path.is_file() and path.suffix.lower() in VALID_EXTENSIONS


def build_augmentation_pipeline(target_size: tuple[int, int]) -> transforms.Compose:
    """Create a stochastic augmentation pipeline for one image size."""
    return transforms.Compose(
        [
            transforms.RandomRotation(degrees=ROTATION_DEGREES),
            transforms.RandomAffine(
                degrees=AFFINE_DEGREES,
                translate=AFFINE_TRANSLATE,
                scale=AFFINE_SCALE,
                shear=AFFINE_SHEAR,
            ),
            transforms.RandomResizedCrop(
                size=target_size,
                scale=RESIZED_CROP_SCALE,
                ratio=RESIZED_CROP_RATIO,
            ),
            transforms.RandomHorizontalFlip(p=H_FLIP_PROBABILITY),
            transforms.ColorJitter(
                brightness=BRIGHTNESS,
                contrast=CONTRAST,
                saturation=SATURATION,
            ),
        ]
    )


def collect_valid_images(folder_path: Path) -> list[Path]:
    """Return sorted image paths, ignoring non-image files safely."""
    files = sorted(folder_path.iterdir(), key=lambda p: p.name.lower())
    return [path for path in files if is_valid_image_file(path)]


def augment_folder_images(input_folder: Path) -> int:
    """Generate 40 augmented images per source image."""
    if not input_folder.exists() or not input_folder.is_dir():
        raise FileNotFoundError(f"Folder not found: {input_folder}")

    source_images = collect_valid_images(input_folder)
    if not source_images:
        raise ValueError(
            f"No valid image files found in folder: {input_folder}\n"
            f"Supported extensions: {', '.join(sorted(VALID_EXTENSIONS))}"
        )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    total_saved = 0

    for source_index, image_path in enumerate(source_images, start=1):
        try:
            with Image.open(image_path) as source:
                original_image = source.convert("RGB")
        except (UnidentifiedImageError, OSError):
            # Skip unreadable files safely even if extension looks valid.
            continue

        pipeline = build_augmentation_pipeline(original_image.size[::-1])

        for aug_index in range(1, NUM_AUGMENTATIONS_PER_IMAGE + 1):
            augmented = pipeline(original_image)
            output_name = f"image_{source_index}_{aug_index}.jpg"
            output_path = OUTPUT_DIR / output_name
            augmented.save(output_path, format="JPEG", quality=JPEG_QUALITY)
            total_saved += 1

    if total_saved == 0:
        raise ValueError("No augmented images were generated. Check that images are readable.")

    print(f"Input folder path: {input_folder.resolve()}")
    print(f"Number of valid source images found: {len(source_images)}")
    print(f"Total augmented images generated: {total_saved}")
    print(f"Output folder path: {OUTPUT_DIR.resolve()}")

    return total_saved


def main() -> None:
    user_input = input("Enter the path to the folder containing images: ").strip()
    if not user_input:
        raise ValueError("No folder path provided. Please enter a valid folder path.")

    augment_folder_images(Path(user_input).expanduser())


if __name__ == "__main__":
    main()